# Member C — Data Transformation

Transforms Coupon Redemption, Customer Churn, and Direct Messaging datasets into canonical schema CSVs.

**Spec:** `docs/notes/data-transformation-member-c.md`

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

REFERENCE_DATE = pd.Timestamp('2024-01-01')
OUTPUT_DIR = '../processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Import from Kaggle

In [ ]:
import kagglehub

coupon_path = kagglehub.dataset_download('vasudeva009/predicting-coupon-redemption')
churn_path = kagglehub.dataset_download('samuelsemaya/e-commerce-customer-churn')
messaging_path = kagglehub.dataset_download('mkechinov/direct-messaging')

print('Coupon:', coupon_path)
print('Churn:', churn_path)
print('Messaging:', messaging_path)

In [ ]:
train = pd.read_csv(f'{coupon_path}/train.csv')
campaign_data = pd.read_csv(f'{coupon_path}/campaign_data.csv')
coupon_item_mapping = pd.read_csv(f'{coupon_path}/coupon_item_mapping.csv')
customer_demographics = pd.read_csv(f'{coupon_path}/customer_demographics.csv')
transactions = pd.read_csv(f'{coupon_path}/customer_transaction_data.csv')
item_data = pd.read_csv(f'{coupon_path}/item_data.csv')

In [ ]:
churn_df = pd.read_csv(f'{churn_path}/data_ecommerce_customer_churn.csv')

In [ ]:
campaigns_msg = pd.read_csv(f'{messaging_path}/campaigns.csv')
messages = pd.read_csv(f'{messaging_path}/messages-demo.csv')
client_first_purchase = pd.read_csv(f'{messaging_path}/client_first_purchase_date.csv')
holidays = pd.read_csv(f'{messaging_path}/holidays.csv')

## 2. Date Alignment

In [ ]:
def align_dates(df, date_cols, ref=REFERENCE_DATE):
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    max_date = df[date_cols].max().max()
    if pd.notna(max_date):
        delta = ref - max_date
        for col in date_cols:
            df[col] = df[col] + delta
    return df

campaign_data = align_dates(campaign_data, ['start_date', 'end_date'])
transactions = align_dates(transactions, ['date'])
campaigns_msg = align_dates(campaigns_msg, ['started_at', 'finished_at'])
messages = align_dates(messages, ['sent_at'])
client_first_purchase = align_dates(client_first_purchase, ['first_purchase_date'])
holidays = align_dates(holidays, ['date'])

## 3. Coupon Redemption → Canonical Tables

In [ ]:
coupon_campaigns = pd.DataFrame({
    'source_campaign_id': 'COUPON_' + campaign_data['campaign_id'].astype(str),
    'campaign_type': campaign_data['campaign_type'],
    'start_date': campaign_data['start_date'].dt.date,
    'end_date': campaign_data['end_date'].dt.date
}).drop_duplicates()

coupon_redemptions = pd.DataFrame({
    'coupon_id': 'COUPON_' + train['coupon_id'].astype(str),
    'campaign_id': 'COUPON_' + train['campaign_id'].astype(str),
    'customer_id': 'COUPON_' + train['customer_id'].astype(str),
    'redemption_status': train['redemption_status'].map({1: True, 0: False})
}).drop_duplicates()

customers_coupon = pd.DataFrame({
    'source_customer_id': 'COUPON_' + transactions['customer_id'].astype(str).unique()
})

products = pd.DataFrame({
    'source_product_id': 'COUPON_' + item_data['item_id'].astype(str),
    'brand': item_data['brand'],
    'category': item_data['category']
}).drop_duplicates()

orders_coupon = pd.DataFrame({
    'source_order_id': [f'COUPON_TXN_{i}' for i in range(len(transactions))],
    'order_date': transactions['date'].dt.date,
    'customer_id': 'COUPON_' + transactions['customer_id'].astype(str)
})

order_items = pd.DataFrame({
    'quantity': transactions['quantity'],
    'sales_value': transactions['selling_price'],
    'other_discount': transactions['other_discount'],
    'coupon_discount': transactions['coupon_discount']
})

## 4. Customer Churn → customers

In [ ]:
customers_churn = pd.DataFrame({
    'source_customer_id': [f'CHURN_{i}' for i in range(len(churn_df))],
    'tenure': churn_df['Tenure'],
    'warehouse_to_home': churn_df['WarehouseToHome'],
    'number_of_devices_registered': churn_df['NumberOfDeviceRegistered'],
    'preferred_order_category': churn_df['PreferedOrderCat'],
    'satisfaction_score': churn_df['SatisfactionScore'],
    'complain': churn_df['Complain'],
    'days_since_last_order': churn_df['DaySinceLastOrder'],
    'cashback_amount': churn_df['CashbackAmount'],
    'churn': churn_df['Churn'].map({1: True, 0: False}),
    'first_purchase_date': (REFERENCE_DATE - pd.to_timedelta(churn_df['Tenure'], unit='D')).dt.date
})

## 5. Direct Messaging → Canonical Tables

In [ ]:
marketing_campaigns = pd.DataFrame({
    'source_campaign_id': 'MSG_' + campaigns_msg['id'].astype(str),
    'campaign_type': campaigns_msg['campaign_type'],
    'channel': campaigns_msg['channel'],
    'topic': campaigns_msg['topic'],
    'started_at': campaigns_msg['started_at'],
    'finished_at': campaigns_msg['finished_at'],
    'total_count': campaigns_msg['total_count']
}).drop_duplicates()

channel_norm_map = {
    'email': 'email',
    'sms': 'sms',
    'web_push': 'push',
    'mobile_push': 'push',
    'whatsapp': 'whatsapp'
}

message_sends = pd.DataFrame({
    'source_message_id': 'MSG_' + messages['message_id'].astype(str),
    'campaign_id': 'MSG_' + messages['campaign_id'].astype(str),
    'customer_id': 'MSG_' + messages['client_id'].astype(str),
    'message_type': messages['message_type'],
    'channel': messages['channel'],
    'channel_norm': messages['channel'].map(channel_norm_map),
    'sent_at': messages['sent_at'],
    'is_opened': messages['is_opened'].astype(bool),
    'is_clicked': messages['is_clicked'].astype(bool),
    'is_unsubscribed': messages['is_unsubscribed'].astype(bool),
    'is_hard_bounced': messages['is_hard_bounced'].astype(bool),
    'is_soft_bounced': messages['is_soft_bounced'].astype(bool),
    'is_complained': messages['is_complained'].astype(bool),
    'is_blocked': messages['is_blocked'].astype(bool),
    'is_purchased': messages['is_purchased'].astype(bool)
})

customers_messaging = pd.DataFrame({
    'source_customer_id': 'MSG_' + client_first_purchase['client_id'].astype(str),
    'first_purchase_date': client_first_purchase['first_purchase_date'].dt.date
}).drop_duplicates()

holidays_out = pd.DataFrame({
    'date': holidays['date'].dt.date,
    'holiday': holidays['holiday']
}).drop_duplicates()

## 6. Merge Customers

In [ ]:
all_customers = pd.concat([customers_coupon, customers_churn, customers_messaging], ignore_index=True)
all_customers = all_customers.drop_duplicates(subset='source_customer_id')
print(f'Total customers: {len(all_customers)}')
print(f'  Coupon: {len(customers_coupon)}')
print(f'  Churn: {len(customers_churn)}')
print(f'  Messaging: {len(customers_messaging)}')

## 7. Write Output CSVs

In [ ]:
def trim_whitespace(df):
    return df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

coupon_campaigns = trim_whitespace(coupon_campaigns)
coupon_redemptions = trim_whitespace(coupon_redemptions)
all_customers = trim_whitespace(all_customers)
products = trim_whitespace(products)
orders_coupon = trim_whitespace(orders_coupon)
order_items = trim_whitespace(order_items)
marketing_campaigns = trim_whitespace(marketing_campaigns)
message_sends = trim_whitespace(message_sends)
holidays_out = trim_whitespace(holidays_out)

coupon_campaigns.to_csv(f'{OUTPUT_DIR}/coupon_campaigns.csv', index=False)
coupon_redemptions.to_csv(f'{OUTPUT_DIR}/coupon_redemptions.csv', index=False)
all_customers.to_csv(f'{OUTPUT_DIR}/customers.csv', index=False)
products.to_csv(f'{OUTPUT_DIR}/products.csv', index=False)
orders_coupon.to_csv(f'{OUTPUT_DIR}/orders.csv', index=False)
order_items.to_csv(f'{OUTPUT_DIR}/order_items.csv', index=False)
marketing_campaigns.to_csv(f'{OUTPUT_DIR}/marketing_campaigns.csv', index=False)
message_sends.to_csv(f'{OUTPUT_DIR}/message_sends.csv', index=False)
holidays_out.to_csv(f'{OUTPUT_DIR}/holidays.csv', index=False)

print('All CSVs written to', OUTPUT_DIR)

## 8. Verification

In [ ]:
print('=== Row Counts ===')
for name, df in [('coupon_campaigns', coupon_campaigns), ('coupon_redemptions', coupon_redemptions),
                  ('customers', all_customers), ('products', products), ('orders', orders_coupon),
                  ('order_items', order_items), ('marketing_campaigns', marketing_campaigns),
                  ('message_sends', message_sends), ('holidays', holidays_out)]:
    print(f'  {name}: {len(df)}')

print('\n=== Date Ranges ===')
print(f'  orders.order_date: {orders_coupon["order_date"].min()} to {orders_coupon["order_date"].max()}')
print(f'  message_sends.sent_at: {message_sends["sent_at"].min()} to {message_sends["sent_at"].max()}')

print('\n=== Channel Values ===')
print(f'  raw: {sorted(message_sends["channel"].unique())}')
print(f'  normalized: {sorted(message_sends["channel_norm"].dropna().unique())}')

print('\n=== ID Prefix Check ===')
prefixes = all_customers['source_customer_id'].str.extract(r'^([A-Z]+)')[0].value_counts()
print(prefixes)